质量太高了，循序渐进，不愧是世界一流大学！

# Exercise 1: Tensor basics 
In this exercise you will learn the basics of tensor creation, manipulation, indexing, broadcasting, vectorization, einsum, and attention masking fundamentals. These basics are important for understanding any complex implementation later on so make sure you understand them well.

**To complete this exercise fill in all TODOs in the functions below.** 

Make sure to check the output of your function and whether or not it fulfills the requirements outlined in the function definition. Do NOT change the function signature or name since we will be running checks on your functions during grading.

### Shape legend used in this notebook
- `B`: batch size
- `T`: sequence length / time
- `D`: feature dimension
- `H`: number of attention heads
- `Dh`: per-head feature dimension

### Debugging tip: what to print
When you get a shape error, print:
- `x.shape`, `x.dtype`, `x.device` 查看形状，数据类型，数据所在设备（cpu，cuda）
- `x.is_contiguous()` (important for `view`) 数据在内存中连续存储，按行优先，view要求数据的内存是连续的，如果tensor不连续，view会报错；要调用 view 之前，先确认 x.is_contiguous() 是 True，否则要先 contiguous() 或改用 reshape。

For masks also print:
- `mask.shape`, `mask.dtype`, `mask.sum()` and a small slice like `mask[0, :10]`

### Reproducibility tip: seeding in PyTorch
Many operations in deep learning involve randomness (e.g., initializing model weights, shuffling data, dropout, random augmentations).（例如，初始化模型权重、打乱数据、dropout、随机增强）。
**Seeding** sets the starting state of PyTorch’s random number generator so that these random choices become **repeatable**.

- If you set the same seed and run the same code again, you should get the same *random* tensors / initial weights.
- If you don’t set a seed, results can vary between runs.

Common usage: `torch.manual_seed(seed)`

Note: even with fixed seeds, some GPU operations can still be non-deterministic due to performance optimizations. For this assignment, seeding is mainly to make debugging easier and to ensure everyone can reproduce the same intermediate results. If you are given a seed, make sure to use it when creating tensors or performing other operations.

## Tensor creation
This warmup exercise teaches you how to create tensors with different shapes and values. A few details about tensor creation that are good to know:
- `torch.tensor([...])` infers dtype from Python values (ints → integer tensor, floats → float tensor).根据python的数据类型转成对应tensor的数据类型
- `torch.arange(start, end)` is **end-exclusive**.不包含末端数值
- `torch.linspace(start, end, steps)` is **end-inclusive**.包含末端数值
- 这两个的区别是第一个按照step递增，最后不一定到达end，第二个是根据steps nums算好step去执行，一定到达end

In [1]:
from collections.abc import Sequence
import torch

In [2]:
def make_tensor(data, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """ Create a tensor from Python data (list/tuple/nested lists（嵌套列表）). """
    # TODO: implement
    return torch.tensor(data,dtype=dtype,device=device)
    raise NotImplementedError

x = make_tensor([[1, 2], [3, 4]], dtype=torch.float32)
print(x)

tensor([[1., 2.],
        [3., 4.]])


In [3]:
def make_zeros(shape: Sequence[int], dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with zeros."""
    # TODO: implement
    return torch.zeros(shape,dtype=dtype,device=device)
    raise NotImplementedError

z = make_zeros((2, 3), dtype=torch.float64)
print(z)

tensor([[0., 0., 0.],
        [0., 0., 0.]], dtype=torch.float64)


In [4]:
def make_ones_like(x: torch.Tensor) -> torch.Tensor:
    """Create a tensor of ones with the same shape, dtype, and device as x. """
    # TODO: implement
    return torch.ones_like(x)
    raise NotImplementedError

base = torch.randn(2, 3, dtype=torch.float32)
print(base)
ones = make_ones_like(base)
print(ones)

tensor([[-0.7153,  1.2767, -1.3009],
        [-0.9319,  0.0077, -1.0067]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])


In [5]:
def make_arange(start: int, end: int, step: int = 1, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a 1D tensor containing values [start, start+step, ..., < end]."""
    # TODO: implement
    # 可以观察到最后 < end
    return torch.arange(start,end,step,dtype=dtype,device=device)
    raise NotImplementedError

ar = make_arange(0, 5, 2, dtype=torch.int64)
print(ar)

tensor([0, 2, 4])


In [6]:
def make_linspace(start: float, end: float, steps: int, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a 1D tensor with evenly spaced values from start to end (inclusive)."""
    # TODO: implement
    return torch.linspace(start,end,steps,dtype=dtype,device=device)
    raise NotImplementedError

ls = make_linspace(0.0, 1.0, steps=5, dtype=torch.float32)
print(ls)
# torch.arange() 和 torch.linspace()的区别是 arange是按照step递进，末端小于end，linspace是在区间内形成steps个点，包括末尾

tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


In [7]:
def make_randn(shape: Sequence[int], seed: int | None = None, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with values from a standard normal distribution."""
    # TODO: implement
    if seed is not None:
        torch.manual_seed(seed=seed) # 设置了seed之后要应用
    return torch.randn(shape,dtype=dtype,device=device)
    raise NotImplementedError

a = make_randn((2, 3), seed=123, dtype=torch.float32)
print(a)

tensor([[-0.1115,  0.1204, -0.3696],
        [-0.2404, -1.1969,  0.2093]])


In [10]:
def cast_dtype_and_move(x: torch.Tensor, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    """Convert tensor dtype and move to device."""
    # TODO: implement
    return x.to(device=device,dtype=dtype)
    raise NotImplementedError

casted = cast_dtype_and_move(torch.tensor([1, 2, 3]), torch.device("cpu"), torch.float32)
print(casted)
print(casted.device)
print(casted.dtype)

tensor([1., 2., 3.])
cpu
torch.float32


## Shape manipulation
Now that we covered the basic tensor creation schemes, we want to focus on shape manipulation. Understanding the difference between these mechanisms is key for building larger systems and many people still get it wrong. 
The core ideas to understand are:
- **Contiguous tensors** store data in a single, row-major memory layout.
- Many ops (especially slicing like `x[:, ::2]`, `transpose`, `permute`) often create **non-contiguous** tensors (no copy but different strides).[第 0 维全取，第 1 维从开头开始、每隔 2 个取一个（步长 2）。]
- `view(...)` is **zero-copy** but typically requires **contiguous** memory → may throw an error.
- `reshape(...)` tries to return a view, but if the tensor is non-contiguous it will **allocate/copy**.
- `contiguous()` forces a contiguous copy when the tensor isn’t contiguous.

If you *need* a view after reordering dims: call `x = x.contiguous()` first (this makes a contiguous copy).

In [11]:
def reshape_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """Reshape tensor to new_shape (may return a view or a copy)."""
    # TODO: implement
    return torch.reshape(x,shape=new_shape)
    raise NotImplementedError

x = torch.arange(6)
print(x)
y = reshape_tensor(x, (2, 3))
print(y)

tensor([0, 1, 2, 3, 4, 5])
tensor([[0, 1, 2],
        [3, 4, 5]])


In [12]:
def view_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """View tensor as new_shape (requires contiguous memory and doesn't allocate new memory for the tensor data)."""
    # TODO: implement
    return x.view(new_shape)
    raise NotImplementedError

y_view = view_tensor(x, (2, 3))
print(y_view)

tensor([[0, 1, 2],
        [3, 4, 5]])


`torch.view` 要求张量在内存中连续且只能改变形状，而 `torch.reshape` 更灵活，必要时会自动复制数据以兼容非连续张量。所以reshape会更通用

In [13]:
def flatten_from_dim(x: torch.Tensor, start_dim: int = 0) -> torch.Tensor:
    """Flatten a tensor starting from start_dim into a single dimension."""
    # TODO: implement
    return torch.flatten(x,start_dim=start_dim)
    raise NotImplementedError

x2 = torch.randn(2, 3, 4)
print(x2.shape)
flat = flatten_from_dim(x2, start_dim=1) # torch.Size([2, 12])
print(flat.shape)

torch.Size([2, 3, 4])
torch.Size([2, 12])


In [14]:
def add_singleton_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Insert a size-1 dimension at position dim."""
    # TODO: implement
    return x.unsqueeze(dim=dim) 
    raise NotImplementedError

x3 = torch.randn(5, 7)
print(x3.shape)
x3s = add_singleton_dim(x3, dim=1)
print(x3s.shape)

torch.Size([5, 7])
torch.Size([5, 1, 7])


In [15]:
def remove_singleton_dims(x: torch.Tensor, dim: int | None = None) -> torch.Tensor:
    """Remove size-1 dimensions."""
    # TODO: implement
    # attention 需要对dim进行讨论
    return x.squeeze() if dim is None else x.squeeze(dim=dim)
    raise NotImplementedError

x4 = torch.randn(2, 1, 3)
print(x4.shape)
x4s = remove_singleton_dims(x4)
print(x4s.shape)

torch.Size([2, 1, 3])
torch.Size([2, 3])


- squeeze是降维，unsqueese是加入维度
- x.squeeze()-去掉所有大小为 1 的维度
- x.squeeze(dim)-只尝试去掉第 dim 维（若它=1）

In [16]:
def transpose_last_two(x: torch.Tensor) -> torch.Tensor:
    """Swap the last two dimensions of x."""
    # TODO: implement
    return x.transpose(-2,-1)
    raise NotImplementedError

x6 = torch.randn(2, 3, 4)
print(x6.shape)
x6t = transpose_last_two(x6)
print(x6t.shape)

torch.Size([2, 3, 4])
torch.Size([2, 4, 3])


In [17]:
def permute_bhwc_to_bchw(x: torch.Tensor) -> torch.Tensor:
    """Convert (B, H, W, C) tensor into (B, C, H, W)."""
    # TODO: implement
    return x.permute(0,3,1,2)
    raise NotImplementedError

x7 = torch.randn(8, 32, 32, 3)
print(x7.shape)
x7p = permute_bhwc_to_bchw(x7)
print(x7p.shape)

torch.Size([8, 32, 32, 3])
torch.Size([8, 3, 32, 32])


transpose 只能调换两个维度，permute 可以重排全部。

In [18]:
def make_contiguous(x: torch.Tensor) -> torch.Tensor:
    """Check if tensor is contiguous and if not make contiguous."""
    # TODO: implement
    return x.contiguous()
    raise NotImplementedError

x8 = torch.randn(4, 6)[:, ::2] # 跳着取值导致不连续
print(x8.is_contiguous())

x8c = make_contiguous(x8)
print(x8c.is_contiguous())

False
True


## Indexing
Now that we know how to create tensors and manipulate them we need to understand how we can extract certain components from them using indexing. 
- Basic slicing (`x[a:b]`) returns a view when possible.
- “Fancy” indexing (lists/tensors of indices) usually allocates a new tensor.
- In-place vs out-of-place matters: if a function says “return a copy, leave the input unchanged”, you need `clone()`.

In [19]:
def slice_rows(x: torch.Tensor, start: int, end: int) -> torch.Tensor:
    """Slice rows in a 2D tensor: x[start:end, :]."""
    # TODO: implement
    return x[start:end,:]
    raise NotImplementedError

x = torch.arange(12).reshape(4, 3)
print(x)
rows = slice_rows(x, 1, 3)
print(rows)

tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])
tensor([[3, 4, 5],
        [6, 7, 8]])


x[start:end,:] attention 不包括end

In [20]:
def select_columns(x: torch.Tensor, cols: Sequence[int]) -> torch.Tensor:
    """Select specific columns from a 2D tensor."""
    # TODO: implement
    return x[:,cols]
    raise NotImplementedError

cols = select_columns(x, [0, 2])
print(cols)

tensor([[ 0,  2],
        [ 3,  5],
        [ 6,  8],
        [ 9, 11]])


In [23]:
def get_diagonal(x: torch.Tensor) -> torch.Tensor:
    """Get the diagonal of a 2D tensor."""
    # TODO: implement
    return x.diagonal()
    raise NotImplementedError

d = get_diagonal(torch.tensor([[1, 2], [3, 4]]))
print(torch.tensor([[1, 2], [3, 4]]))
print(torch.tensor([[1, 2], [3, 4]]).shape)
print(d)
print(d.shape) # 不是[2,1]也不是[1,2]

tensor([[1, 2],
        [3, 4]])
torch.Size([2, 2])
tensor([1, 4])
torch.Size([2])


In [25]:
def set_subtensor(x: torch.Tensor, row_idx: int, col_idx: int, value: float) -> torch.Tensor:
    """Return a copy of x where x[row_idx, col_idx] is set to value."""
    # TODO: implement
    out = x.clone() # 明确指出了需要copy
    out[row_idx,col_idx]=value
    return out
    raise NotImplementedError

base = torch.zeros(2, 2)
print(base)
out = set_subtensor(base, 0, 1, 5.0) # 把在(0,1)位置的base设置为5.0
print(out)

tensor([[0., 0.],
        [0., 0.]])
tensor([[0., 5.],
        [0., 0.]])


In [27]:
def gather_rows(x: torch.Tensor, row_indices: torch.Tensor) -> torch.Tensor:
    """Gather (concat) rows from x using row_indices."""
    # TODO: implement
    return x[row_indices] # fancy indices
    raise NotImplementedError

x2 = torch.tensor([[10, 11], [20, 21], [30, 31]])
idx = torch.tensor([2, 0])
# advanced indexing
gathered = gather_rows(x2, idx) # 按照顺序检索出第二行和第〇行
print(x2)
print(idx)
print(gathered) 

tensor([[10, 11],
        [20, 21],
        [30, 31]])
tensor([2, 0])
tensor([[30, 31],
        [10, 11]])


## Broadcasting and reducing
Now we're covering a pytorch mechanism that lets you apply elementwise ops without using python loops. It's important to understand how it works to trace your shapes in complicated systems. The broadcasting rules to know are:
- Dimensions align from the **right**. 两个张量的形状右对齐，短的那个左边补 1：
- A dimension can broadcast if it’s equal or one of them is **1**.某一维能广播，当且仅当相等或其中一个是 1

### Reduction ops and `keepdim`

When you reduce over a dimension (e.g. `sum`, `mean`, `max`), PyTorch can either:

- **remove** the reduced dimension (`keepdim=False`, default), or
- **keep** it as size 1 (`keepdim=True`)

归约（sum/mean/max…）会把某一维"压掉"。
keepdim=False（默认）→ 那一维消失；keepdim=True → 那一维变成 1。
保留成 1 是为了后面能和原张量直接广播，不用手动 unsqueeze。

Keeping the dimension is often helpful because it makes broadcasting back “just work”.
不然需要手动 unsqueeze 才能广播回来，易错
#### Shape diagram examples
真的太形象了，简直是最好的pytorch教程！

Assume `x` has shape `(B, T, D)`:

**Sum over time**
- `x.sum(dim=1)` → shape `(B, D)`
- `x.sum(dim=1, keepdim=True)` → shape `(B, 1, D)`

**Mean over features**
- `x.mean(dim=2)` → shape `(B, T)`
- `x.mean(dim=2, keepdim=True)` → shape `(B, T, 1)`

#### Why `keepdim=True` helps with broadcasting

Example: center `x` by subtracting the mean over `T`

- If `m = x.mean(dim=1)` has shape `(B, D)`, then `x - m` **fails** (shapes `(B,T,D)` and `(B,D)` don't align).
- If `m = x.mean(dim=1, keepdim=True)` has shape `(B,1,D)`, then `x - m` **works** via broadcasting.

In [28]:
def sum_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Sum tensor values along dimension dim."""
    # TODO: implement
    return x.sum(dim=dim,keepdim=keepdim)
    raise NotImplementedError

x = torch.ones(2, 3)
print(x)
y = sum_over_dim(x, dim=1)
print(y.shape)

tensor([[1., 1., 1.],
        [1., 1., 1.]])
torch.Size([2])


In [30]:
def mean_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Mean along dimension dim."""
    # TODO: implement
    return x.mean(dim=dim,keepdim=keepdim)
    raise NotImplementedError

x2 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(x2)
y2 = mean_over_dim(x2, dim=0)
print(y2.shape)
print(y2)

tensor([[1., 2.],
        [3., 4.]])
torch.Size([2])
tensor([2., 3.])


In [35]:
def max_over_dim(x: torch.Tensor, dim: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Max values and argmax indices along dimension dim."""
    # TODO: implement
    result = x.max(dim=dim)
    return result.values,result.indices
    raise NotImplementedError

x3 = torch.tensor([[1.0, 5.0], [3.0, 2.0]])
print(x3)
print(x3.shape)
values, idx = max_over_dim(x3, dim=1)
print(values)
print(idx)

tensor([[1., 5.],
        [3., 2.]])
torch.Size([2, 2])
tensor([5., 3.])
tensor([1, 0])


- dim=1 不是指"沿着第1维方向求最大值"，而是指"把第1维压缩掉，在这一维上求最大值"。
- dim=k 所有下标都固定，只有第 k 个下标在变化，对变化的那一维取 max。
- 这有点反直觉，不过也正是因为规定定义才避免了歧义

In [37]:
def argmax_over_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Argmax indices along dimension dim."""
    # TODO: implement
    return x.argmax(dim=dim)
    raise NotImplementedError
print(x3)
idx2 = argmax_over_dim(x3, dim=1) # 只有dim=1动，其他维度的dim不动，因此求的是dim=1的最大值索引，反直觉但避免了歧义
print(idx2)

tensor([[1., 5.],
        [3., 2.]])
tensor([1, 0])


In [40]:
def broadcast_add_vector(x: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Add a vector v to each row of a 2D tensor x using broadcasting."""
    # TODO: implement
    return x+v
    raise NotImplementedError

x4 = torch.zeros(3, 2)
print(x4)
print(x4.shape)
v = torch.tensor([10.0, 20.0]) # shape (2,) 所以在广播的时候自动变成(1,2) 再变成(3,2),我们并不能说v是行向量还是列向量
print(v)
print(v.shape) # v.shape 的维度是2，并不是[2,1]或者[1,2]也是为了避免歧义，因此才有了broadcasting时候维度右对齐规定
y4 = broadcast_add_vector(x4, v)
print(y4)

# 3 2
# . 2

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])
torch.Size([3, 2])
tensor([10., 20.])
torch.Size([2])
tensor([[10., 20.],
        [10., 20.],
        [10., 20.]])


## Vectorization
We want to avoid slow (due to per-iteration overhead) python loops as much as possible and pytorch gives us many tools to avoid it. We cover these basics:
- `cat` vs `stack` (concatenate existing dims vs create a new dim)
- `repeat` vs `expand`
- `scatter_add` / `index_add` for accumulation
- `where` for conditional selection

### `expand` vs `repeat`

- `repeat(...)` **copies** data → larger tensor with independent storage.
- `expand(...)` **does not copy** data → it creates a *view* with clever strides.

This has two important implications:

1) `expand` only works when expanding a **size-1 dimension** (broadcasting a singleton).
2) The expanded tensor may have **many positions pointing to the same memory**.  
   Modifying the expanded tensor can therefore produce surprising results (multiple rows change).所以一般不会改变 expand后的数值，只会用于计算

Rule of thumb: # 经验法则
- Use `expand` for read-only broadcasting.
- Use `repeat` if you truly need independent copies.


NOTE: We implore（恳求） you to write your own quick checks from now on for calling the functions and checking their output. As before you are still required to fill in the TODOs in each function.

In [ ]:
def concat_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    """Concatenate tensors along dim. NOTE: This will always allocate new memory"""
    # TODO: implement
    return torch.cat(tensors=tensors,dim=dim)
    raise NotImplementedError

x1 = torch.ones(2,3)
x2 = torch.ones(2,3)
print(x1)
print(x2)
a = concat_tensors((x1,x2),dim=1) # (2,3) ,(2,3) -> (2,6) 因为 dim = 1变化，其他维度不变，也就是堆叠到dim=1上
print(a)
print(a.shape)

tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])
torch.Size([2, 6])


In [45]:
def stack_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    """Stack tensors along a new dimension dim."""
    # TODO: implement
    return torch.stack(tensors=tensors,dim=dim)
    raise NotImplementedError
a =stack_tensors((x1,x2),dim=1)
print(a)
print(a.shape)

tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.]]])
torch.Size([2, 2, 3])


- torch.cat 常用于拼特征 / 通道 x = torch.cat([feat_a, feat_b], dim=1)
- torch.stack 用于把多张图片摞成一个 batch
- imgs = [img1, img2, img3]        # 每个 (3, 224, 224)
- batch = torch.stack(imgs, dim=0) # (3, 3, 224, 224)  ← batch 维

In [47]:
def repeat_tensor(x: torch.Tensor, repeats: Sequence[int]) -> torch.Tensor:
    """Repeat tensor along each dimension."""
    # TODO: implement
    return x.repeat(*repeats)
    raise NotImplementedError

x = torch.tensor([[1., 2., 3.]])   # 形状 (1, 3)
r = repeat_tensor(x,(4, 2))     # 参数是“重复次数”：第0维重复4次，第1维1次
print(r.shape)         # torch.Size([4, 3])
print(r)
print(r.storage().size())  # 12  ← 真的存了 12 个数

torch.Size([4, 6])
tensor([[1., 2., 3., 1., 2., 3.],
        [1., 2., 3., 1., 2., 3.],
        [1., 2., 3., 1., 2., 3.],
        [1., 2., 3., 1., 2., 3.]])
24


In [48]:
def expand_tensor(x: torch.Tensor, *sizes: int) -> torch.Tensor:
    """Expand tensor to a larger size without copying data.(Sizes can be -1 to keep original dimension.)"""
    # TODO: implement
    return x.expand(*sizes)
    raise NotImplementedError

e = expand_tensor(x,(4, 3))     # 参数是“目标形状”：直接写目标 (4, 3)
print(e.shape)         # torch.Size([4, 3])
print(e)
print(e.storage().size())  # 3  ← 底层只存了 3 个数！

torch.Size([4, 3])
tensor([[1., 2., 3.],
        [1., 2., 3.],
        [1., 2., 3.],
        [1., 2., 3.]])
3


- repeat 真的复制出一份新数据，expand 只造个假象共享同一块内存。
- 只读的时候，expand 和 repeat 输出看起来一样，但 expand 不占内存。
- 要改的时候，只有 repeat 能用，expand 会报错（除非 .clone()）。
- repeat 的参数是重复次数，expand 的参数是目标形状

In [50]:
def cumsum_over_dim(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    """Cumulative sum along dim."""
    # TODO: implement
    return x.cumsum(dim=dim)
    raise NotImplementedError

# dim=k，就是让第 k 个下标递增的那个方向,tensor[2,3]那么dim=0沿着2这个下标累加
# dim=k的通用理解或者设计原理就是 指哪打哪，谁被指定，谁就发生改变，其余不变
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(x)
print(cumsum_over_dim(x, dim=0))
print(cumsum_over_dim(x, dim=1))

tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2, 3],
        [5, 7, 9]])
tensor([[ 1,  3,  6],
        [ 4,  9, 15]])


In [52]:
def where_select(mask: torch.Tensor, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Elementwise select: return a where mask is True else b. mask must be broadcastable to a and b."""
    # TODO: implement
    return torch.where(mask,a,b)
    raise NotImplementedError

mask = torch.tensor([True, False, True, False])
a = torch.tensor([1., 2., 3., 4.])
b = torch.tensor([10., 20., 30., 40.])
print(where_select(mask, a, b))

# 测试广播
mask2 = torch.tensor([[True], [False], [True]]) # (3,1),所以广播到(3,4),也就是第二维度扩展
a2 = torch.zeros(3, 4)
b2 = torch.ones(3, 4)
print(where_select(mask2, a2, b2))

tensor([ 1., 20.,  3., 40.])
tensor([[0., 0., 0., 0.],
        [1., 1., 1., 1.],
        [0., 0., 0., 0.]])


In [ ]:
def one_hot(indices: torch.Tensor, num_classes: int, dtype: torch.dtype | None = None) -> torch.Tensor:
    """
    Create one-hot encodings.
    Output is a tensor of the same shape as indices with an added dimension of size num_classes at the end, 
    where the value along that dimension is 1 if it matches the index and 0 otherwise.

    Shapes:
    - indices: (...,) integer tensor
    Return:
    - out: (..., num_classes)

    Requirements:
    - Must work for arbitrary leading shape.
    - No Python loops.
    """
    # TODO: implement
    # F.one_hot() 只接受 long 类型的 indices
    out =torch.nn.functional.one_hot(indices.to(dtype=torch.long),num_classes=num_classes)
    return out if dtype is None else out.to(dtype=dtype)
    raise NotImplementedError

idx = torch.tensor([[0, 1, 2],
                    [3, 2, 0]])# shape (2,3)
out = one_hot(idx, 4) # 生成[2,3,4], 和索引值相等的位置为 1,也就是索引 0，1，2位置为1, 矩阵的数值决定了0出现在哪里
print(out) # [2,3,:]中1出现的位置就是[2,3]的index数值

tensor([[[1, 0, 0, 0],
         [0, 1, 0, 0],
         [0, 0, 1, 0]],

        [[0, 0, 0, 1],
         [0, 0, 1, 0],
         [1, 0, 0, 0]]])


In [57]:
def scatter_add_1d(
    values: torch.Tensor, indices: torch.Tensor, size: int
) -> torch.Tensor:
    """
    Sum `values` into an output vector at positions `indices`.

    Shapes:
    - values: (N,)
    - indices: (N,) integer indices in [0, size)
    Return:
    - out: (size,) with same dtype and device as values

    Requirement:
    - no Python loops
    """
    # TODO: implement
    out = torch.zeros(size,dtype=values.dtype,device=values.device)
    return out.scatter_add(0,indices,values)
    raise NotImplementedError

v = torch.tensor([10., 20., 30., 40.])
i = torch.tensor([1, 0, 1, 3])
print(scatter_add_1d(v, i, 4))

tensor([20., 40.,  0., 40.])


In [ ]:
def batched_token_histogram(tokens: torch.Tensor, vocab_size: int) -> torch.Tensor:
    """
    Count token occurrences per batch item.

    Shapes:
    - tokens: (B, T) int64
    Return:
    - counts: (B, vocab_size) where counts[b, v] = number of times token v appears in tokens[b] 

    Requirements:
    - No Python loops over B or T.
    """
    # TODO: implement
    # 这个需要借用上一步的scatter_add，非常巧妙实现
    B = tokens.shape[0]
    out = torch.zeros(B,vocab_size, dtype=tokens.dtype, device=tokens.device)
    return out.scatter_add(1,tokens,torch.ones_like(tokens))
    raise NotImplementedError

tokens = torch.tensor([[0, 1, 1, 2],
                       [2, 2, 2, 0]]) #shape (2,4) - (B,T) 查一查token出现的次数，加起来 
vocab_size = 3 #0,1,2, 去tokens里面找dim=1维度的1，2，3出现次数
counts = batched_token_histogram(tokens, vocab_size)
print(counts)

tensor([[1, 2, 1],
        [1, 0, 3]])


In [61]:
def masked_mean(x: torch.Tensor, mask: torch.Tensor, dim: int) -> torch.Tensor:
    """
    Mean over `dim` considering only mask==True entries.

    Convention:
    - mask: bool tensor broadcastable to x
    - mask==True means "keep this entry"

    Return: same shape as x.mean(dim=dim)

    Requirements:
    - Avoid division by zero: if all mask are False along `dim`, define mean as 0.
    """
    # TODO: implement
    keep = torch.broadcast_to(mask, x.shape)
    selected = torch.where(keep,x,torch.zeros_like(x))
    total = selected.sum(dim=dim)
    count = keep.sum(dim=dim)
    return total/count.clamp_min(1) # clamp_min(a) 把张量里所有小于 a 的值，抬到 a，因为可能出现0/0情况，就变成了0/1=0
    raise NotImplementedError

x = torch.tensor([[1.,  2.,  3.,  4.],
                  [5.,  6.,  7.,  8.]])
mask = torch.tensor([[True, False, True, False],
                     [True, True,  True, True]])
# 这里的 dim = 1是沿着dim=1求mean 
print(masked_mean(x, mask, 1))       # tensor([2., 6.5])   <- (1+3)/2, (5+6+7+8)/4=6.5

tensor([2.0000, 6.5000])


## Einsum warmup
Now that you’re comfortable with shapes and broadcasting, we’ll introduce `torch.einsum`, a concise way to express tensor operations by explicitly naming axes and summing over repeated indices.

### The idea
You describe each input tensor by labeling its dimensions with letters, e.g.
- `x: (B, T, D)` → `"btd"`
- `W: (D, H)`    → `"dh"`

Then you tell einsum what output labels you want:
- `"btd,dh->bth"`

### Rules of einsum
1) **Same letter = same axis** (must match in size, except broadcastable size-1).
2) **Repeated letters are summed over** (a “contraction”).
3) **Letters that appear in the output are kept** (in that order).
4) You can **reorder axes** just by changing the output label order.
   
1. 同字母 = 同一根轴：同一个字母出现在不同输入里，表示它们沿这根轴对齐（该维长度必须相等；长度为 1 的可以广播）。
2. 重复出现的字母会被求和：某个字母在输出里没出现、只在输入里出现，就沿这根轴做求和（即"收缩"掉）。
3. 输出里出现的字母会被保留：最终结果的维度就是输出下标里列出的这些字母，按你写的顺序排列。
4. 改输出字母顺序 = 重新排列轴：想转置/换轴，只要在输出标注里调整字母顺序即可，不用额外调 transpose。

### Tiny cheat sheet
- Sum over an axis: `"btd->bt"` (sums over `d`)
- Transpose: `"ij->ji"`
- Dot product: `"d,d->"` or batched `"btd,btd->bt"`
- Matrix multiply: `"ik,kj->ij"`
- Batched matmul: `"bij,bjk->bik"`
- Outer product: `"i,j->ij"`

### How to derive an einsum (recommended workflow)
1) Write down shapes with named axes (e.g. `q: b h t d`, `k: b h s d`).
2) Decide which axes you want to **sum over** (give them the same letter in both inputs).
3) Decide which axes you want to **keep** in the output (write them after `->`).

In this section, you’ll use einsum to implement building blocks that show up in attention:
- linear projections (`x @ W`)
- dot products
- attention score matrices (`QKᵀ`)
- applying attention weights (`softmax(scores) @ V`)

NOTE: For these exercises you are required to use `torch.einsum` not `matmul` (we check). You are also not required to understand the attention mechanism at this point and the exercises are sovable without. It is good however, to remember the implementations in this exercise for future implementations.

In [62]:
def einsum_linear_btd_dh_to_bth(x: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    """
    Linear projection using einsum.

    Shapes:
    - x: (B, T, D)
    - W: (D, H)
    Return:
    - y: (B, T, H)
    """
    # TODO: implement
    return torch.einsum("btd,dh->bth",x,W)
    raise NotImplementedError

In [63]:
def einsum_pairwise_dot(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Pairwise dot product between x and y.

    Shapes:
    - x: (B, T, D)
    - y: (B, T, D)
    Return:
    - dots: (B, T) where dots[b,t] = dot(x[b,t], y[b,t])
    """
    # TODO: implement
    return torch.einsum("btd,btd->bt",x,y)
    raise NotImplementedError

In [64]:
def einsum_qk_scores(q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """
    Compute attention scores QK^T using einsum.

    Shapes:
    - q: (B, H, T, Dh)
    - k: (B, H, T, Dh)
    Return:
    - scores: (B, H, T, T) where scores[b,h,i,j] = dot(q[b,h,i], k[b,h,j])
    """
    # TODO: implement
    return torch.einsum("bhid,bhjd->bhij",q,k)
    raise NotImplementedError

In [65]:
def einsum_apply_attention(weights: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Apply attention weights to values using einsum.

    Shapes:
    - weights: (B, H, T, T)
    - v:       (B, H, T, Dh)
    Return:
    - out:     (B, H, T, Dh) where out[b,h,i] = sum_j weights[b,h,i,j] * v[b,h,j]
    """
    # TODO: implement
    return torch.einsum("bhij,bhjd->bhid",weights,v)
    raise NotImplementedError

In [67]:
import torch

# ---------- 1. Linear ----------
B, T, D, H = 2, 3, 4, 5
x = torch.randn(B, T, D)
W = torch.randn(D, H)
y = einsum_linear_btd_dh_to_bth(x, W)
print("linear        : x", tuple(x.shape), "W", tuple(W.shape), "-> y", tuple(y.shape))

# ---------- 2. Pairwise dot ----------
x2 = torch.randn(B, T, D)
y2 = torch.randn(B, T, D)
dots = einsum_pairwise_dot(x2, y2)
print("pairwise_dot  : x", tuple(x2.shape), "y", tuple(y2.shape), "-> dots", tuple(dots.shape))

# ---------- 3. QK scores ----------
q = torch.randn(B, H, T, D)
k = torch.randn(B, H, T, D)
scores = einsum_qk_scores(q, k)
print("qk_scores     : q", tuple(q.shape), "k", tuple(k.shape), "-> scores", tuple(scores.shape))

# ---------- 4. Apply attention ----------
weights = torch.softmax(torch.randn(B, H, T, T), dim=-1)
v = torch.randn(B, H, T, D)
out = einsum_apply_attention(weights, v)
print("apply_attn    : w", tuple(weights.shape), "v", tuple(v.shape), "-> out", tuple(out.shape))

linear        : x (2, 3, 4) W (4, 5) -> y (2, 3, 5)
pairwise_dot  : x (2, 3, 4) y (2, 3, 4) -> dots (2, 3)
qk_scores     : q (2, 5, 3, 4) k (2, 5, 3, 4) -> scores (2, 5, 3, 3)
apply_attn    : w (2, 5, 3, 3) v (2, 5, 3, 4) -> out (2, 5, 3, 4)


## Attention Fundamentals
This exercise introduces some building blocks of the attention mechanism which we will encounter extensively throughout the course. It's not yet required for you to fully understand the mechanism to implement the exercises. However, it's good to remember these building blocks for the future. 

To complete the exercises you should familiarize yourself with these topics:
- Stable softmax read: https://jaykmody.com/blog/stable-softmax/
- Masking: typically this means setting masked logits to -inf *before* softmax. 掩码在softmax之前
- For attention: causal masks are upper-triangular (no attending to the future).因果注意力就是现在的只能关注当前和之前的，不能看到未来

In [68]:
def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Numerically stable softmax along `dim`.

    Requirements:
    - Must not overflow for large values in x.
    - Output sums to 1 along `dim`.
    """
    # TODO: implement
    shifted = x - x.amax(dim=dim,keepdim=True)
    exp_x = shifted.exp()
    return exp_x / exp_x.sum(dim=dim,keepdim=True)
    
    raise NotImplementedError

In [69]:
def masked_fill_tensor(x: torch.Tensor, mask: torch.Tensor, value: float) -> torch.Tensor:
    """
    Return a copy of x where positions with mask == True are replaced by `value`.
    
    Requirements:
    - mask must be broadcastable to x.
    - do NOT modify x in-place.
    """
    # TODO: implement
    return x.masked_fill(mask=mask,value=value)
    raise NotImplementedError

In [ ]:
def masked_softmax(x: torch.Tensor, mask: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Softmax over x with a boolean mask.

    Convention:
    - mask == True means "invalid and must receive probability 0".
    - Do masking before softmax (i.e., set invalid logits to a large negative).”

    Requirements:
    - Must be numerically stable.
    - Output must be exactly 0 where mask==True.
    - If all entries are masked along `dim`, return all zeros along `dim`.
    - You may reuse functions you implemented above.
    """
    # TODO: implement using masked_fill_tensor + stable_softmax
    masked_x = masked_fill_tensor(x,mask,float("-inf"))
    all_masked = torch.broadcast_to(mask,x.shape).all(dim=dim,keepdim=True) # 检测是否全为 True，如果是的话需要特殊标记
    safe_x = torch.where(all_masked,torch.zeros_like(masked_x),masked_x) # 如果是全True，填充0
    probs = stable_softmax(safe_x,dim=dim)
    return masked_fill_tensor(probs,mask,0.0)
    raise NotImplementedError


In [ ]:
def make_causal_mask(T: int, device: torch.device | str | None = None) -> torch.Tensor:
    """
    Create a causal (future-masking) boolean mask of shape (T, T).

    Convention:
    - mask[i, j] == True  => position (i attends to j) is NOT allowed (j is in the future)
    - mask[i, j] == False => allowed

    So this is an upper-triangular mask above the diagonal.

    Return:
    - mask: boolean tensor on the specified device

    Example (T=4):
        [[F, T, T, T],
         [F, F, T, T],
         [F, F, F, T],
         [F, F, F, F]]
    """
    # TODO: implement
    return torch.ones((T,T),dtype=torch.bool,device=device).triu(diagonal=1) # 上三角向上平移1个单位
    raise NotImplementedError

In [72]:
def apply_causal_mask(attn_logits: torch.Tensor, value: float = -1e9) -> torch.Tensor:
    """
    Apply a causal mask to attention logits.

    Expected shapes:
    - attn_logits: (..., T, T)

    Returns:
    - masked logits (same shape) where masked positions have been set to `value`.

    Notes:
    - Create a causal mask for the final two dims.
    - Broadcast it across leading dims.
    - You may reuse functions declared above.
    """
    # TODO: implement using make_causal_mask + masked_fill_tensor
    T = attn_logits.shape[-1]
    casual_mask=make_causal_mask(T,device=attn_logits.device)
    return masked_fill_tensor(attn_logits,casual_mask,value)
    raise NotImplementedError

In [74]:
import torch

# 1. stable_softmax
x = torch.tensor([[1000.0, 1001.0, 1002.0]])
p = stable_softmax(x)
print("stable_softmax:", p)
print("概率和:", p.sum(dim=-1))
print("参考结果:", torch.softmax(x, dim=-1))

# 2. masked_fill_tensor
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
mask = torch.tensor([False, True, False])
print("\nmasked_fill_tensor:\n", masked_fill_tensor(x, mask, -1.0))
print("原输入（应保持不变）:\n", x)

# 3. masked_softmax
x = torch.tensor([[1000.0, 1001.0, 1002.0], [1.0, 2.0, 3.0]])
mask = torch.tensor([[False, True, False], [True, True, True]])
p = masked_softmax(x, mask)
print("\nmasked_softmax:\n", p)
print("预期：第一行约 [0.1192, 0, 0.8808]，第二行全 0")
print("每行概率和:", p.sum(dim=-1))

# 4. make_causal_mask
print("\nmake_causal_mask:\n", make_causal_mask(3))
print("预期：仅对角线上方为 True")

# 5. apply_causal_mask
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0],
                  [7.0, 8.0, 9.0]])
print("\napply_causal_mask:\n", apply_causal_mask(x, value=-100.0))
print("预期：仅对角线上方替换为 -100")

stable_softmax: tensor([[0.0900, 0.2447, 0.6652]])
概率和: tensor([1.])
参考结果: tensor([[0.0900, 0.2447, 0.6652]])

masked_fill_tensor:
 tensor([[ 1., -1.,  3.],
        [ 4., -1.,  6.]])
原输入（应保持不变）:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])

masked_softmax:
 tensor([[0.1192, 0.0000, 0.8808],
        [0.0000, 0.0000, 0.0000]])
预期：第一行约 [0.1192, 0, 0.8808]，第二行全 0
每行概率和: tensor([1.0000, 0.0000])

make_causal_mask:
 tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])
预期：仅对角线上方为 True

apply_causal_mask:
 tensor([[   1., -100., -100.],
        [   4.,    5., -100.],
        [   7.,    8.,    9.]])
预期：仅对角线上方替换为 -100
